In [1]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELL 1: Clone repo & install dependencies                           ║
# ╚══════════════════════════════════════════════════════════════════════╝

!rm -rf EveFL
!git clone https://github.com/Major-Project-EveFL/EveFL.git
!cd EveFL && pip install -q -e .

Cloning into 'EveFL'...
remote: Enumerating objects: 135, done.
remote: Counting objects: 100% (135/135), done.
remote: Compressing objects: 100% (103/103), done.
remote: Total 135 (delta 65), reused 92 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (135/135), 95.95 KiB | 13.71 MiB/s, done.
Resolving deltas: 100% (65/65), done.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for evefl (pyproject.toml) ... done


In [2]:
!cd EveFL && pip install -q -r requirements.txt
!cd EveFL && pip install -q -e .

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.9/41.9 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 103.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 767.5/767.5 kB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 342.3/342.3 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 465.1/465.1 kB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 906.4/906.4 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 114.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 103.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.

In [3]:
!pip uninstall -y tensorflow tensorflow-io-gcs-filesystem

Found existing installation: tensorflow 2.20.0
Uninstalling tensorflow-2.20.0:
  Successfully uninstalled tensorflow-2.20.0
Found existing installation: tensorflow-io-gcs-filesystem 0.37.1
Uninstalling tensorflow-io-gcs-filesystem-0.37.1:
  Successfully uninstalled tensorflow-io-gcs-filesystem-0.37.1


In [4]:
import torch, qiskit
import importlib.metadata

# Check flwr version WITHOUT triggering the simulation/TensorFlow import chain
flwr_ver = importlib.metadata.version("flwr")
print(f"flwr: {flwr_ver}")
print(f"torch: {torch.__version__}")
print(f"qiskit: {qiskit.__version__}")

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  PATCH: Fix circular import in flwr 1.11.1                           ║
# ╚══════════════════════════════════════════════════════════════════════╝

import flwr.server.superlink.state.in_memory_state as _ims
import flwr.server.superlink.state.sqlite_state as _ss

# Patch in_memory_state.py: move validate_task_ins_or_res into methods
_ims_source = _ims.__file__

with open(_ims_source, "r") as f:
    code = f.read()

# Remove top-level import
code = code.replace(
    "from flwr.server.utils import validate_task_ins_or_res\n",
    ""
)

# Inject local imports inside store_task_ins and store_task_res
code = code.replace(
    '        # Validate task\n        errors = validate_task_ins_or_res(task_ins)',
    '        from flwr.server.utils import validate_task_ins_or_res  # lazy import\n        # Validate task\n        errors = validate_task_ins_or_res(task_ins)',
    1  # only first occurrence (store_task_ins)
)

code = code.replace(
    '        # Validate task\n        errors = validate_task_ins_or_res(task_res)',
    '        from flwr.server.utils import validate_task_ins_or_res  # lazy import\n        # Validate task\n        errors = validate_task_ins_or_res(task_res)',
    1  # only first occurrence (store_task_res)
)

with open(_ims_source, "w") as f:
    f.write(code)

# Patch sqlite_state.py: same fix
_ss_source = _ss.__file__

with open(_ss_source, "r") as f:
    code = f.read()

code = code.replace(
    "from flwr.server.utils.validator import validate_task_ins_or_res\n",
    ""
)

code = code.replace(
    '        # Validate task\n        errors = validate_task_ins_or_res(task_ins)',
    '        from flwr.server.utils.validator import validate_task_ins_or_res  # lazy import\n        # Validate task\n        errors = validate_task_ins_or_res(task_ins)',
    1
)

code = code.replace(
    '        # Validate task\n        errors = validate_task_ins_or_res(task_res)',
    '        from flwr.server.utils.validator import validate_task_ins_or_res  # lazy import\n        # Validate task\n        errors = validate_task_ins_or_res(task_res)',
    1
)

with open(_ss_source, "w") as f:
    f.write(code)

print("✅ Patched flwr 1.11.1 circular import")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELL 2: Locate dataset & partition                                  ║
# ╚══════════════════════════════════════════════════════════════════════╝

import os
from pathlib import Path

# Kaggle dataset path (adjust if your input folder name differs)
KAGGLE_DATA = Path("/kaggle/input/datasets/nih-chest-xrays/data")
if not KAGGLE_DATA.exists():
    # Fallback: search common Kaggle paths
    candidates = [
        Path("/kaggle/input/datasets/nih-chest-xrays/data"),
        Path("/kaggle/input/nih-chest-xray"),
        Path("/kaggle/input/chestxray14"),
        Path("/kaggle/input/chest-xray14"),
        Path("/kaggle/input/nih-chest-xray14"),
    ]
    for c in candidates:
        if c.exists():
            KAGGLE_DATA = c
            break

assert KAGGLE_DATA.exists(), f"Dataset not found. Searched: {candidates}"
print(f"Dataset found at: {KAGGLE_DATA}")

PARTITION_ROOT = Path("/kaggle/working/partitions")
RESULTS_DIR = Path("/kaggle/working/results")

In [ ]:
from pathlib import Path
import sys

# EveFL repo
sys.path.insert(0, "/kaggle/working/EveFL")

# Actual Kaggle dataset
KAGGLE_DATA = Path("/kaggle/input/datasets/nih-chest-xrays/data")

# Working directories
PARTITION_ROOT = Path("/kaggle/working/partitions")
RESULTS_DIR = Path("/kaggle/working/results")

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Dataset:", KAGGLE_DATA)
print("Dataset exists:", KAGGLE_DATA.exists())
print("Partitions:", PARTITION_ROOT)
print("Results:", RESULTS_DIR)

In [ ]:
# from chatgpt
from pathlib import Path

RESULTS_DIR = Path("/kaggle/working/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

PARTITION_ROOT = Path("/kaggle/working/partitions")

print("RESULTS_DIR:", RESULTS_DIR)
print("PARTITION_ROOT:", PARTITION_ROOT)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELL 4: Run all 4 Eve scenarios                                     ║
# ╚══════════════════════════════════════════════════════════════════════╝

from evefl.fl.server import run_experiment

SCENARIOS = [
    ("A_no_eve", 0.00),
]

NUM_ROUNDS = 10          # paper: 50
LOCAL_EPOCHS = 1        # paper: 5
BATCH_SIZE = 32          # paper: 32
N_QUBITS = 64          # paper: 1024

all_summaries = []

for name, intercept in SCENARIOS:
    print(f"\n{'='*60}")
    print(f"Running scenario: {name} (intercept={intercept})")
    print(f"{'='*60}")

    output_path = RESULTS_DIR / f"{name}.json"

    summary = run_experiment(
        data_root=KAGGLE_DATA,
        partition_root=PARTITION_ROOT,
        num_clients=3,
        num_rounds=NUM_ROUNDS,
        local_epochs=LOCAL_EPOCHS,
        batch_size=BATCH_SIZE,
        n_qubits=N_QUBITS,
        intercept_probability=intercept,
        seed=42,
        experiment_name=name,
        output_path=output_path,
    )
    all_summaries.append(summary)
    print(f"Done: {output_path}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELL 5: Combine & export                                            ║
# ╚══════════════════════════════════════════════════════════════════════╝

import json

combined = {
    "meta": {
        "num_clients": 3,
        "num_rounds": NUM_ROUNDS,
        "local_epochs": LOCAL_EPOCHS,
        "batch_size": BATCH_SIZE,
        "n_qubits": N_QUBITS,
        "subset_fraction": SUBSET_FRACTION,
    },
    "scenarios": all_summaries,
}

combined_path = RESULTS_DIR / "combined_results.json"
with open(combined_path, "w") as f:
    json.dump(combined, f, indent=2)

print(f"\nCombined results: {combined_path}")

# Summary table
print(f"\n{'Scenario':<15} {'SECURE':<8} {'CAUTION':<8} {'LOCKDOWN':<8} {'Time(s)':<10}")
print("-" * 55)
for s in all_summaries:
    print(f"{s['experiment']:<15} {s['secure_rounds']:<8} {s['caution_rounds']:<8} {s['lockdown_rounds']:<8} {s['elapsed_seconds']:<10.1f}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELL 6: Download results                                        ║
# ╚══════════════════════════════════════════════════════════════════════╝

from IPython.display import FileLink

for f in RESULTS_DIR.glob("*.json"):
    display(FileLink(str(f)))

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELL 5.5: Inline visualization (presentation-ready)                 ║
# ╚══════════════════════════════════════════════════════════════════════╝

import json
import matplotlib.pyplot as plt
import numpy as np

# Load combined results
with open(RESULTS_DIR / "combined_results.json") as f:
    data = json.load(f)

scenarios = data["scenarios"]
names = [s["experiment"] for s in scenarios]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("EveFL: Quantum-Aware FL Security", fontsize=16, fontweight="bold")

# 1. Stacked bar — state distribution
ax = axes[0, 0]
secure = [s["secure_rounds"] for s in scenarios]
caution = [s["caution_rounds"] for s in scenarios]
lockdown = [s["lockdown_rounds"] for s in scenarios]

bottom1 = np.array(secure)
bottom2 = bottom1 + np.array(caution)

ax.bar(names, secure, label="SECURE", color="#10b981")
ax.bar(names, caution, bottom=bottom1, label="CAUTION", color="#f59e0b")
ax.bar(names, lockdown, bottom=bottom2, label="LOCKDOWN", color="#ef4444")
ax.set_ylabel("Rounds")
ax.set_title("State Distribution by Scenario")
ax.legend()

# 2. Timing
ax = axes[0, 1]
times = [s["elapsed_seconds"] for s in scenarios]
bars = ax.bar(names, times, color=["#6366f1", "#8b5cf6", "#a855f7", "#d946ef"])
ax.set_ylabel("Seconds")
ax.set_title("Experiment Duration")
for bar, t in zip(bars, times):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f"{t:.0f}s", ha="center", va="bottom", fontsize=9)

# 3. Security score (higher = more secure)
ax = axes[1, 0]
total = np.array(secure) + np.array(caution) + np.array(lockdown)
security_pct = np.array(secure) / total * 100
ax.barh(names, security_pct, color="#10b981")
ax.set_xlabel("% SECURE Rounds")
ax.set_title("Security Score")
ax.set_xlim(0, 100)
for i, v in enumerate(security_pct):
    ax.text(v + 2, i, f"{v:.1f}%", va="center")

# 4. Threat response efficiency
ax = axes[1, 1]
# Caution + Lockdown = "response rounds" (not just blind FedAvg)
response = np.array(caution) + np.array(lockdown)
ax.bar(names, response, color="#f59e0b", label="CAUTION")
ax.bar(names, lockdown, bottom=caution, color="#ef4444", label="LOCKDOWN")
ax.set_ylabel("Rounds")
ax.set_title("Adaptive Response Rounds")
ax.legend()

plt.tight_layout()
plt.savefig(RESULTS_DIR / "presentation_summary.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Saved: {RESULTS_DIR / 'presentation_summary.png'}")